# Extraction Test Notebook
Validate point time series extraction from the preprocessed Landsat collection.

**Steps**
1. Init GEE & build collection
2. Extract single point, single year — LST + cloud flag
3. Extract single point, multi year — LST + cloud flag



## 1. Init GEE & build collection

In [1]:
%load_ext autoreload
%autoreload 2

import ee
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# Anchor all paths to the project root (one level up from notebooks/)
ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd().parent

from rs_timeseries.config import load_config, get_ee_project
from rs_timeseries.io import build_aoi, build_merged_collection
from rs_timeseries.preprocessing import preprocess_collection
from rs_timeseries.extraction import (
    extract_point_timeseries,
    extract_multipoint_timeseries,
    load_points_from_geojson
)

# Load config and init GEE
config = load_config(ROOT / "configs" / "swiss_alps_lst.yaml")

project_id = get_ee_project()
ee.Initialize(project=project_id) if project_id else ee.Initialize()
print(f"GEE initialized. ROOT: {ROOT}")

GEE initialized. ROOT: d:\Projects\landsat-timeseries


## 2. Extract single point — LST + cloud flag


In [ ]:
config["masking"]["remove_overlap"] = False
config["masking"]["apply_cloud_mask"] = False

LON, LAT = 9.15, 47.17   # AMD2
bands = ["SurfT", "cloud", "Red"]

aoi = build_aoi(config)
raw_collection = build_merged_collection(config, aoi)
collection = preprocess_collection(raw_collection, config)

collection1 = collection.select(bands)

df_lst = extract_point_timeseries(
    collection=collection1,
    lon=LON,
    lat=LAT,
    bands=bands,
    scale=30,
    start_year=1998,
    end_year=1998,
    verbose=True,
)

In [ ]:
df_lst.head()

## 3. Extract single point, multi year — LST + cloud flag

In [ ]:
config["masking"]["remove_overlap"] = False
config["masking"]["apply_cloud_mask"] = False

LON, LAT = 9.15, 47.17   # AMD2
bands = ["SurfT", "cloud"]

aoi = build_aoi(config)
raw_collection = build_merged_collection(config, aoi)
collection = preprocess_collection(raw_collection, config)

collection1 = collection.select(bands)

df_lst_yrs = extract_point_timeseries(
    collection=collection1,
    lon=LON,
    lat=LAT,
    bands=bands,
    scale=30,
    start_year=1998,
    end_year=2000,
    verbose=True,
)

In [ ]:
df_lst_yrs.head()

## 4. Extract multi point, multi year — LST + cloud flag

In [2]:
points = load_points_from_geojson(
    geojson_path=ROOT / "data" / "raw" / "stations" / "imis" / "imis_stations.geojson", 
    point_id_col="station",
    keep_cols=None # ["utmX", "utmY"]
)

Loaded 124 points using 'station'


In [ ]:
bands = ['SurfT', 'cloud']

config["masking"]["remove_overlap"] = False
config["masking"]["apply_cloud_mask"] = False

aoi = build_aoi(config)
raw_collection = build_merged_collection(config, aoi)
collection = preprocess_collection(raw_collection, config)

collection1 = collection.select(bands)

# Extract all stations → individual CSVs + merged
df_multi = extract_multipoint_timeseries(
    collection=collection1,
    points=points[:5],
    bands=["SurfT", "cloud"],
    output_folder=ROOT / "outputs" / "imis_stations_landsat", 
    scale=30,
    start_year=1984,
    end_year=2022,
    variable="lst",
    save_merged=False
)

print(f"\nExtraction complete!")
print(df_multi.groupby("point_id").size())


--- LST: ALI2 [6.99, 46.49] ---
  1984: 20 scenes
  1985: 18 scenes
  1986: 22 scenes
  1987: 29 scenes
  1988: 29 scenes
  1989: 32 scenes
  1990: 35 scenes
  1991: 26 scenes
  1992: 27 scenes
  1993: 26 scenes
  1994: 28 scenes
  1995: 29 scenes
  1996: 29 scenes
  1997: 28 scenes
  1998: 32 scenes
  1999: 47 scenes
  2000: 51 scenes
  2001: 58 scenes
  2002: 54 scenes
  2003: 36 scenes
  2004: 40 scenes
  2005: 35 scenes
  2006: 37 scenes
  2007: 27 scenes
  2008: 33 scenes
  2009: 40 scenes
  2010: 36 scenes
  2011: 42 scenes
  2012: 22 scenes
  2013: 61 scenes
  2014: 77 scenes
  2015: 89 scenes
  2016: 83 scenes
  2017: 82 scenes
  2018: 84 scenes
  2019: 89 scenes
  2020: 58 scenes
  2021: 52 scenes
  2022: 60 scenes

Extracted 1703 scenes (1984–2022) | clear: 1703 cloudy: 0
Saved 1703 scenes → d:\Projects\landsat-timeseries\outputs\imis_stations_landsat\ALI2_lst_1984-2022.csv

--- LST: AMD2 [9.15, 47.17] ---
  1984: 7 scenes
  1985: 17 scenes
  1986: 12 scenes
  1987: 26 scene

In [ ]:
df_multi